# Phase Plane Analysis of Three-Species Cross-Feeding Model

**Jian Wang**  
*January 2026*

---

## Overview

This notebook presents a comprehensive phase plane analysis of a three-species microbial community model featuring:

- **S-specialist**: Substrate specialists that grow on primary substrate
- **M-specialist**: Metabolite specialists that depend on metabolic byproducts
- **G-specialist (Generalist)**: Can utilize both pathways with weighting parameter ω

The model captures bidirectional cross-feeding, competition, and synergistic interactions in a framework that generalizes classic ecological theory to metabolic cooperation.

### Model Equations

$$\frac{dN_S}{dt} = r_S \cdot N_S \left[1 + \sigma_{SM} \cdot \frac{N_M}{K_M} + (1-\omega) \cdot \sigma_{SG} \cdot \frac{N_G}{K_G} - \omega \cdot \alpha_{SG} \cdot \frac{N_G}{K_G}- \frac{N_S}{K_S}\right]$$

$$\frac{dN_M}{dt} = r_M \cdot N_M \left[-1 + \sigma_{MS} \cdot \frac{N_S}{K_S} + \omega \cdot \sigma_{MG} \cdot \frac{N_G}{K_G} - (1-\omega) \cdot \alpha_{MG} \cdot \frac{N_G}{K_G} - \frac{N_M}{K_M}\right]$$

$$\frac{dN_G}{dt} = r_G \cdot N_G \left[\omega \left(1 - \alpha_{GS} \cdot\frac{N_S}{K_S}+\sigma_{GM}\cdot\frac{N_M}{K_M}\right) + (1-\omega) \left(-1 - \alpha_{GM} \cdot \frac{N_M}{K_M}+\sigma_{GS}\frac{N_S}{K_S}\right) - \frac{N_G}{K_G}\right]$$

---

In [ ]:
# Import required packages
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

from three_species_model import ThreeSpeciesModel
from phase_plane_analysis import PhasePlaneAnalyzer

# Set plotting style
sns.set_context('paper', font_scale=1.5)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

%matplotlib inline

## 1. Model Initialization and Parameter Exploration

First, let's initialize the model with biologically realistic parameters and examine the baseline dynamics.

In [ ]:
# Initialize model with default parameters
model = ThreeSpeciesModel()

# Display parameters
print("Model Parameters:")
print("="*60)
print("\nGrowth rates:")
print(f"  r_S = {model.params['r_S']:.3f} (S-specialist)")
print(f"  r_M = {model.params['r_M']:.3f} (M-specialist)")
print(f"  r_G = {model.params['r_G']:.3f} (Generalist)")

print("\nCarrying capacities:")
print(f"  K_S = {model.params['K_S']:.1f}")
print(f"  K_M = {model.params['K_M']:.1f}")
print(f"  K_G = {model.params['K_G']:.1f}")

print("\nSynergistic coefficients (cooperation):")
print(f"  σ_SM = {model.params['sigma_SM']:.3f} (S benefits from M)")
print(f"  σ_MS = {model.params['sigma_MS']:.3f} (M benefits from S)")
print(f"  σ_SG = {model.params['sigma_SG']:.3f} (S benefits from G)")
print(f"  σ_GS = {model.params['sigma_GS']:.3f} (G benefits from S)")
print(f"  σ_MG = {model.params['sigma_MG']:.3f} (M benefits from G)")
print(f"  σ_GM = {model.params['sigma_GM']:.3f} (G benefits from M)")

print("\nCompetition coefficients:")
print(f"  α_SG = {model.params['alpha_SG']:.3f} (S-G competition)")
print(f"  α_MG = {model.params['alpha_MG']:.3f} (M-G competition)")
print(f"  α_GS = {model.params['alpha_GS']:.3f} (G-S competition)")
print(f"  α_GM = {model.params['alpha_GM']:.3f} (G-M competition)")

print("\nPathway weighting:")
print(f"  ω = {model.params['omega']:.3f} (0 = metabolite pathway, 1 = substrate pathway)")
print("="*60)

## 2. Equilibrium Analysis

A crucial first step in phase plane analysis is identifying all equilibrium points and determining their stability. This tells us the long-term outcomes of the system.

In [ ]:
# Find all equilibria
equilibria = model.find_equilibria(n_attempts=100)

print(f"Found {len(equilibria)} equilibrium points:\n")
print("="*80)

for i, eq in enumerate(equilibria):
    print(f"\nEquilibrium {i+1}: N_S={eq[0]:.3f}, N_M={eq[1]:.3f}, N_G={eq[2]:.3f}")
    
    # Ecological classification
    eco_class = model.classify_equilibrium_ecology(eq)
    print(f"  Ecological state: {eco_class}")
    
    # Stability analysis
    stability = model.stability_analysis(eq)
    print(f"  Stability: {stability['type']}")
    print(f"  Eigenvalues: {stability['eigenvalues']}")
    
    if stability['stable']:
        print("  ✓ This is a STABLE equilibrium (attracts nearby trajectories)")
    else:
        print("  ✗ This is an UNSTABLE equilibrium (repels nearby trajectories)")
    
    print("-"*80)

### Biological Interpretation

**Key questions to address:**
1. Can all three species coexist at a stable equilibrium?
2. What are the conditions for coexistence?
3. How does the pathway weighting (ω) affect community composition?
4. Which species pairs can coexist in the absence of the third?

**Expected patterns (Gore lab style):**
- M-specialist cannot survive alone (obligate cross-feeder, note the -1 in its equation)
- S-specialist can survive independently (grows on substrate)
- Generalist survival depends on ω (pathway balance)
- Three-species coexistence requires balanced cooperation and competition

## 3. Time Series Dynamics

Before examining phase planes, let's visualize how populations change over time from different initial conditions.

In [ ]:
# Define diverse initial conditions
initial_conditions = [
    np.array([80.0, 10.0, 10.0]),   # S-dominated
    np.array([10.0, 80.0, 10.0]),   # M-dominated
    np.array([10.0, 10.0, 80.0]),   # G-dominated
    np.array([50.0, 50.0, 50.0]),   # Equal abundance
    np.array([30.0, 30.0, 30.0]),   # Equal, lower density
]

# Create phase plane analyzer
analyzer = PhasePlaneAnalyzer(model)

# Plot time series
fig = analyzer.plot_timeseries(initial_conditions, t_max=100)
plt.savefig('../figures/timeseries_dynamics.png', dpi=300, bbox_inches='tight')
plt.show()

print("Time series show convergence to equilibrium from different initial conditions.")
print("Notice how different starting compositions can lead to different final states!")

## 4. Phase Plane Analysis: 2D Projections

Since our system is 3-dimensional, we analyze 2D phase planes by projecting onto pairs of species. This is the classic approach in ecology (e.g., Lotka-Volterra analysis).

### 4.1 S-M Phase Plane (Generalist at intermediate density)

In [ ]:
# S-M phase plane with G fixed at moderate level
fig = analyzer.plot_phase_portrait_2D(
    species_pair=(0, 1),  # S vs M
    fixed_species_val=50.0,  # G = 50
    initial_conditions=[
        np.array([80.0, 10.0, 50.0]),
        np.array([10.0, 80.0, 50.0]),
        np.array([50.0, 50.0, 50.0]),
        np.array([20.0, 20.0, 50.0]),
        np.array([100.0, 10.0, 50.0]),
    ],
    t_max=100,
    grid_range=(0, 120),
    n_grid=20,
    show_nullclines=True,
    show_equilibria=True
)

plt.savefig('../figures/phase_plane_SM.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("- Blue dashed line: S-specialist nullcline (dN_S/dt = 0)")
print("- Red dashed line: M-specialist nullcline (dN_M/dt = 0)")
print("- Intersections of nullclines = equilibria")
print("- Arrows show direction of population change")
print("- Trajectories (colored lines) flow toward stable equilibria (green stars)")

### 4.2 S-G Phase Plane (M-specialist at intermediate density)

In [ ]:
# S-G phase plane with M fixed
fig = analyzer.plot_phase_portrait_2D(
    species_pair=(0, 2),  # S vs G
    fixed_species_val=50.0,  # M = 50
    initial_conditions=[
        np.array([80.0, 50.0, 10.0]),
        np.array([10.0, 50.0, 80.0]),
        np.array([50.0, 50.0, 50.0]),
        np.array([100.0, 50.0, 20.0]),
    ],
    t_max=100,
    grid_range=(0, 120),
    n_grid=20
)

plt.savefig('../figures/phase_plane_SG.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nS-G interaction depends strongly on ω:")
print(f"  Current ω = {model.params['omega']:.2f}")
print("  High ω → G uses substrate pathway → competes with S")
print("  Low ω → G uses metabolite pathway → cooperates with S")

### 4.3 M-G Phase Plane (S-specialist at intermediate density)

In [ ]:
# M-G phase plane with S fixed
fig = analyzer.plot_phase_portrait_2D(
    species_pair=(1, 2),  # M vs G
    fixed_species_val=50.0,  # S = 50
    initial_conditions=[
        np.array([50.0, 80.0, 10.0]),
        np.array([50.0, 10.0, 80.0]),
        np.array([50.0, 50.0, 50.0]),
        np.array([50.0, 20.0, 100.0]),
    ],
    t_max=100,
    grid_range=(0, 120),
    n_grid=20
)

plt.savefig('../figures/phase_plane_MG.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nM-G interaction also depends on ω:")
print("  High ω → G benefits from M's metabolites (cooperation)")
print("  Low ω → G competes with M for S's metabolites")

## 5. Three-Dimensional Phase Space

To visualize the full 3D dynamics, we plot trajectories in the complete phase space.

In [ ]:
# 3D phase space visualization
initial_conditions_3d = [
    np.array([80.0, 10.0, 10.0]),
    np.array([10.0, 80.0, 10.0]),
    np.array([10.0, 10.0, 80.0]),
    np.array([50.0, 50.0, 50.0]),
    np.array([70.0, 20.0, 30.0]),
    np.array([30.0, 60.0, 40.0]),
]

fig = analyzer.plot_3D_phase_space(
    initial_conditions_3d,
    t_max=100,
    show_equilibria=True,
    elev=20,
    azim=45
)

plt.savefig('../figures/phase_space_3D.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n3D phase space shows:")
print("  • Trajectories from different initial conditions")
print("  • Convergence to stable equilibria (green stars)")
print("  • Basin of attraction for coexistence state")
print("  • Circle markers = initial conditions")
print("  • Triangle markers = final states")

## 6. Bifurcation Analysis: The Role of Pathway Weighting (ω)

The parameter ω determines the generalist's metabolic strategy:
- ω = 0: Pure metabolite pathway specialist
- ω = 1: Pure substrate pathway specialist
- 0 < ω < 1: True generalist using both pathways

Let's explore how community composition changes with ω.

In [ ]:
# Perform bifurcation analysis
print("Running bifurcation analysis across ω values...")
print("This may take a minute...\n")

bifurcation_data = analyzer.bifurcation_analysis_omega(
    omega_range=(0.0, 1.0),
    n_omega=50,
    N0=np.array([50.0, 50.0, 50.0])
)

# Plot bifurcation diagram
fig = analyzer.plot_bifurcation_diagram(bifurcation_data)
plt.savefig('../figures/bifurcation_omega.png', dpi=300, bbox_inches='tight')
plt.show()

# Analyze coexistence region
coexist_omega = bifurcation_data['omega'][bifurcation_data['coexistence']]

if len(coexist_omega) > 0:
    print(f"\nThree-species coexistence occurs for:")
    print(f"  ω ∈ [{coexist_omega.min():.3f}, {coexist_omega.max():.3f}]")
    print(f"\nThis represents {100*len(coexist_omega)/len(bifurcation_data['omega']):.1f}% of parameter space.")
else:
    print("\nNo three-species coexistence found in this parameter range.")
    print("Consider adjusting cooperation/competition parameters.")

### Biological Interpretation of Bifurcation

**Expected patterns:**
1. **Low ω (metabolite pathway)**: Generalist behaves like M-specialist
   - May outcompete M-specialist due to flexibility
   - Cooperates with S-specialist

2. **High ω (substrate pathway)**: Generalist behaves like S-specialist
   - May outcompete S-specialist
   - Cooperates with M-specialist

3. **Intermediate ω**: Balanced strategy
   - Generalist uses both pathways
   - Enables three-species coexistence
   - Creates niche differentiation

**Key insight from Gore lab perspective:**  
The generalist's metabolic flexibility (ω) acts as a **tunable niche parameter** that can either promote coexistence (intermediate ω) or lead to competitive exclusion (extreme ω values).

## 7. Sensitivity Analysis: Competition vs. Cooperation

Let's explore how the balance between competition (α) and cooperation (σ) affects coexistence.

In [ ]:
# Scan over cooperation strength
sigma_values = np.linspace(0.1, 0.8, 10)
alpha_values = np.linspace(0.1, 0.8, 10)

# Store results
coexistence_map = np.zeros((len(sigma_values), len(alpha_values)))

# Save original parameters
original_sigma_SM = model.params['sigma_SM']
original_alpha_SG = model.params['alpha_SG']

print("Scanning cooperation-competition parameter space...\n")

for i, sigma in enumerate(sigma_values):
    for j, alpha in enumerate(alpha_values):
        # Update parameters (scale all cooperation/competition uniformly)
        model.params['sigma_SM'] = sigma
        model.params['sigma_MS'] = sigma * 1.2
        model.params['sigma_SG'] = sigma * 0.6
        model.params['sigma_GS'] = sigma * 0.8
        model.params['alpha_SG'] = alpha
        model.params['alpha_MG'] = alpha
        
        # Simulate
        sol = model.simulate(np.array([50.0, 50.0, 50.0]), (0, 500))
        final_state = np.array([sol['N_S'][-1], sol['N_M'][-1], sol['N_G'][-1]])
        
        # Check coexistence
        if np.all(final_state > 1.0):
            coexistence_map[i, j] = 1

# Restore original parameters
model.params['sigma_SM'] = original_sigma_SM
model.params['alpha_SG'] = original_alpha_SG

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(coexistence_map, origin='lower', cmap='RdYlGn',
               extent=[alpha_values.min(), alpha_values.max(),
                      sigma_values.min(), sigma_values.max()],
               aspect='auto')

ax.set_xlabel('Competition strength (α)', fontsize=14)
ax.set_ylabel('Cooperation strength (σ)', fontsize=14)
ax.set_title('Three-Species Coexistence Region', fontsize=16)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Coexistence (1) vs Exclusion (0)', fontsize=12)

# Add diagonal line where σ = α
ax.plot([alpha_values.min(), alpha_values.max()],
        [sigma_values.min(), sigma_values.max()],
        'b--', linewidth=2, label='σ = α (balanced)')

ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()

plt.savefig('../figures/coexistence_map.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCoexistence occurs in {100*coexistence_map.mean():.1f}% of parameter combinations.")
print("\nKey finding: Cooperation must exceed competition (σ > α) for robust coexistence!")

## 8. Summary and Ecological Insights

### Main Findings

1. **Equilibrium Structure**:
   - Multiple equilibria exist, including extinction, two-species, and three-species states
   - Stability depends on parameter values and initial conditions
   - M-specialist is an obligate cross-feeder (cannot survive alone)

2. **Phase Plane Topology**:
   - Nullclines intersect to create equilibria
   - Vector fields show basins of attraction
   - Trajectories converge to stable equilibria from diverse initial states

3. **Role of Pathway Weighting (ω)**:
   - Intermediate ω promotes three-species coexistence
   - Extreme ω values lead to competitive exclusion
   - ω acts as a niche differentiation parameter

4. **Cooperation-Competition Balance**:
   - Strong cooperation (high σ) favors coexistence
   - Strong competition (high α) leads to exclusion
   - Critical threshold: σ > α typically required

### Connection to Experimental Systems

This model framework applies to:
- **Syntrophic microbial communities** (e.g., anaerobic digesters)
- **Cross-feeding in biofilms** (spatial structure + metabolic exchange)
- **Engineered consortia** (designed cooperation for biotechnology)
- **Natural communities** (soil microbiomes, gut microbiota)

### Future Directions

1. Add **spatial structure** (reaction-diffusion PDEs or agent-based models)
2. Include **evolutionary dynamics** (adaptive ω, mutation-selection)
3. Incorporate **environmental fluctuations** (varying substrate, temperature)
4. Extend to **N-species generalization** with metabolic networks
5. Compare with **experimental data** from chemostat or microfluidic devices

---

**References:**
- Gore et al. (2009) "Snowdrift game dynamics and facultative cheating in yeast" *Nature*
- Momeni et al. (2013) "Using artificial systems to explore the ecology and evolution of symbioses" *CELS*
- Goldford et al. (2018) "Emergent simplicity in microbial community assembly" *Science*
- Estrela et al. (2022) "Metabolic rules of microbial community assembly" *Nature Ecology & Evolution*

In [ ]:
print("Analysis complete! ✓")
print("\nAll figures saved to: ../figures/")
print("\nFigures generated:")
print("  1. timeseries_dynamics.png - Population dynamics over time")
print("  2. phase_plane_SM.png - S-M phase portrait")
print("  3. phase_plane_SG.png - S-G phase portrait")
print("  4. phase_plane_MG.png - M-G phase portrait")
print("  5. phase_space_3D.png - 3D phase space trajectories")
print("  6. bifurcation_omega.png - Bifurcation diagram for ω")
print("  7. coexistence_map.png - Cooperation-competition phase diagram")